# 03 — Transfer gap analysis

Answers the project's two questions, and assembles the error sample for hand-coding.

Regenerate every figure and table with:

```bash
python scripts/make_report.py --model baseline
```

In [ ]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import pandas as pd, numpy as np
pd.set_option("display.width", 200)
from wcb.logging_utils import setup_logging
setup_logging(logging.WARNING)

In [ ]:
from wcb import analysis, config as C, data, features
MODEL = "baseline"   # switch to "roberta" once the GPU sweep has run
gaps = analysis.transfer_gap_table(MODEL, n_boot=C.N_BOOTSTRAP)
summary = analysis.average_over_seeds(gaps)
frames = data.load_annotated(C.PRIMARY_SEED)

## Q1 — how much worse on an unseen bank?

In [ ]:
cols = ["pooled_weighted_f1", "lobo_weighted_f1", "gap_total", "gap_total_std"]
summary[cols].round(4).head(10)

## Q2 — what predicts the gap?

**n = 25, so this is descriptive, not inferential.** The feature set was fixed in advance
and is reported whole, flat features included. Confidence intervals are bootstrapped over
banks and are wide by construction — that width is the finding.

In [ ]:
feat = features.compute_bank_features(frames, include_embeddings=False).dropna(axis=1, how="all")
analysis.correlate_features_with_gap(summary, feat, "gap_total", n_boot=C.N_BOOTSTRAP).round(4)

## Error analysis

50 LOBO errors, stratified by bank, preferring sentences the **pooled** model got right —
those isolate the cost of the bank being unseen from sentences that are simply hard.

Fill in `error_category` by hand: *new vocabulary* / *bank-specific convention* /
*genuinely ambiguous*.

In [ ]:
errors = analysis.sample_lobo_errors(MODEL, C.PRIMARY_SEED, frames, n=50)
print(f"{len(errors)} errors; {errors.pooled_correct.sum()} of them the pooled model got right")
errors[["bank", "true_label", "lobo_label", "pooled_label", C.TEXT_COLUMN]].head(12)

In [ ]:
errors.to_csv(C.RESULTS_DIR / f"{MODEL}_error_sample.csv", index=False)
print("saved for hand-coding")